# LandslideRisk-48: reference solution
Each week the crew has 12 hours for inspections at the 12 new sites. An inspection costs the
site's `inspection_hours`; inspecting a site on a day it is relevant is worth 15 hours. Plans
run in date order and an inspection that no longer fits in the week's hours is skipped.
Plan: (1) load data, (2) build features from each reading and *earlier* readings at the same
site, (3) estimate calibrated probabilities with folds grouped by site, (4) compare causal
decision rules on the weekly net-value metric, (5) refit and plan the test weeks.
The rule used here is causal: a site is inspected on day d based only on readings up to day d
and a threshold fixed in advance.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 48
CATCH_VALUE = 15.0
WEEKLY_BUDGET = 12.0
TARGET = "landslide_risk_48h"
DATA = Path("./dataset/public")
OUT = Path("./working")
OUT.mkdir(parents=True, exist_ok=True)

# Queries (train.csv / test.csv): one row per week. train.csv lists the relevant site-days as
# "location_id@YYYY-MM-DD". Readings (*_readings.csv): one row per site and day.
train_queries = pd.read_csv(DATA / "train.csv")
test_queries = pd.read_csv(DATA / "test.csv")
train = pd.read_csv(DATA / "train_readings.csv", parse_dates=["timestamp"])
test = pd.read_csv(DATA / "test_readings.csv", parse_dates=["timestamp"])
sample = pd.read_csv(DATA / "sample_submission.csv")
print(train_queries.shape, test_queries.shape, train.shape, test.shape, sample.shape)

relevant = {(int(t.split("@")[0]), t.split("@")[1]) for cell in train_queries.site_days for t in cell.split()}
train["day"] = train.timestamp.dt.strftime("%Y-%m-%d")
test["day"] = test.timestamp.dt.strftime("%Y-%m-%d")
train[TARGET] = [int((s, d) in relevant) for s, d in zip(train.location_id, train.day)]
print("train sites:", train.location_id.nunique(), "| test sites:", test.location_id.nunique(),
      "| overlap:", len(set(train.location_id) & set(test.location_id)),
      "| positive rate:", round(train[TARGET].mean(), 4))

## Metric
Same as the grader: run each week's plan in date order under the weekly budget, sum the net
value, and divide by the net value of inspecting exactly the relevant site-days in date order.

In [ ]:
def run_week(rows):
    """rows: one week's planned inspections, already in execution order."""
    left, net = WEEKLY_BUDGET, 0.0
    for hours, rel in zip(rows.inspection_hours.to_numpy(), rows[TARGET].to_numpy()):
        if hours <= left + 1e-9:
            left -= hours
            net += (CATCH_VALUE if rel else 0.0) - hours
    return net

def weekly_score(frame, inspect, priority):
    """frame: readings with query_id, day, inspection_hours, TARGET. Within a day, higher
    priority is executed first."""
    plan = frame.assign(_p=priority)[np.asarray(inspect, dtype=bool)]
    plan = plan.sort_values(["day", "_p"], ascending=[True, False])
    ref = frame[frame[TARGET] == 1].sort_values(["day", "location_id"])
    net = sum(run_week(g) for _, g in plan.groupby("query_id"))
    best = sum(run_week(g) for _, g in ref.groupby("query_id"))
    return min(1.0, max(0.0, net / best))

## Features
Each site's rows are processed separately, in timestamp order, and only the current and
earlier readings are used. Train and test sites never mix, so concatenating the two frames
only saves code. `location_id` is not a model feature because every test site is new.

In [ ]:
RAIN = ["rainfall_1h_mm", "rainfall_6h_mm", "rainfall_24h_mm", "rainfall_72h_mm"]

def build_features(df):
    df = df.sort_values(["location_id", "timestamp"]).copy()
    g = df.groupby("location_id")

    # Fill small sensor gaps with the site's previous reading (past only).
    for c in RAIN + ["soil_moisture", "vegetation_index", "temperature_c"]:
        df[c + "_ffill"] = g[c].ffill()

    doy = df.timestamp.dt.dayofyear
    df["doy_sin"], df["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    df["aspect_sin"], df["aspect_cos"] = np.sin(np.radians(df.aspect_deg)), np.cos(np.radians(df.aspect_deg))

    r1, r6, r24, r72 = (df[c + "_ffill"] for c in RAIN)
    df["rain_1_6h"], df["rain_6_24h"], df["rain_24_72h"] = r6 - r1, r24 - r6, r72 - r24
    df["rain_24_over_72"] = r24 / (r72 + 1.0)
    df["slope_x_moist"] = df.slope_deg * df.soil_moisture_ffill
    df["slope_x_rain72"] = df.slope_deg * r72
    df["moist_x_rain72"] = df.soil_moisture_ffill * r72
    df["sin_slope"] = np.sin(np.radians(df.slope_deg))

    # Antecedent wetness from earlier days at the same site.
    g = df.groupby("location_id")
    for c in ["rainfall_72h_mm_ffill", "soil_moisture_ffill"]:
        df[c + "_lag1"] = g[c].shift(1)
        df[c + "_lag3"] = g[c].shift(3)
        df[c + "_d1"] = df[c] - df[c + "_lag1"]
    df["rain24_sum7d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(7, min_periods=1).sum())
    df["rain24_sum30d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(30, min_periods=1).sum())
    df["moist_mean7d"] = g["soil_moisture_ffill"].transform(lambda s: s.rolling(7, min_periods=1).mean())

    df["soil_type"] = df.soil_type.astype(pd.CategoricalDtype(["sandy", "loam", "clay", "silt", "rocky", "peat"]))
    return df

full = build_features(pd.concat([train.assign(_split="train"), test.assign(_split="test")], ignore_index=True))
tr = full[full._split == "train"].reset_index(drop=True)
te = full[full._split == "test"].reset_index(drop=True)

BASE = ["elevation_m", "slope_deg", "aspect_deg", "soil_moisture", "soil_type", "vegetation_index",
        *RAIN, "temperature_c"]
ENGINEERED = BASE + ["doy_sin", "doy_cos", "aspect_sin", "aspect_cos", "rain_1_6h", "rain_6_24h",
        "rain_24_72h", "rain_24_over_72", "slope_x_moist", "slope_x_rain72", "moist_x_rain72", "sin_slope",
        "rainfall_72h_mm_ffill_lag1", "rainfall_72h_mm_ffill_lag3", "rainfall_72h_mm_ffill_d1",
        "soil_moisture_ffill_lag1", "soil_moisture_ffill_lag3", "soil_moisture_ffill_d1",
        "rain24_sum7d", "rain24_sum30d", "moist_mean7d"]

## Validation that mirrors the test set
Three folds grouped by site: each validation fold holds ~13 unseen sites, close to the 12 test
sites, so the weekly budget is shared by a similar number of candidates.

In [ ]:
folds = list(GroupKFold(n_splits=3).split(tr, groups=tr.location_id))

LGB_PARAMS = dict(objective="binary", learning_rate=0.02, n_estimators=2000, num_leaves=4,
                  min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                  reg_lambda=5.0, random_state=SEED, verbose=-1)

def make_lr(features):
    num = [c for c in features if c != "soil_type"]
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("soil", OneHotEncoder(handle_unknown="ignore"), ["soil_type"]),
    ])
    return make_pipeline(prep, LogisticRegression(max_iter=3000))

def oof_lr(features):
    oof = np.zeros(len(tr))
    for f, v in folds:
        oof[v] = make_lr(features).fit(tr.loc[f, features], tr.loc[f, TARGET]).predict_proba(tr.loc[v, features])[:, 1]
    return oof

best_iters = []
def oof_lgb(features):
    oof = np.zeros(len(tr))
    for f, v in folds:
        m = lgb.LGBMClassifier(**LGB_PARAMS)
        m.fit(tr.loc[f, features], tr.loc[f, TARGET], eval_set=[(tr.loc[v, features], tr.loc[v, TARGET])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        best_iters.append(m.best_iteration_)
        oof[v] = m.predict_proba(tr.loc[v, features])[:, 1]
    return oof

def fold_score(inspect, priority):
    inspect, priority = np.asarray(inspect, dtype=bool), np.asarray(priority, dtype=float)
    return float(np.mean([weekly_score(tr.loc[v], inspect[v], priority[v]) for _, v in folds]))

oof = {"LR raw": oof_lr(BASE), "LR engineered": oof_lr(ENGINEERED), "LightGBM engineered": oof_lgb(ENGINEERED)}
oof["blend LR raw + LightGBM"] = 0.5 * oof["LR raw"] + 0.5 * oof["LightGBM engineered"]
y = tr[TARGET].to_numpy()
for name, p in oof.items():
    pc = np.clip(p, 1e-6, 1 - 1e-6)
    print(f"{name:26s} OOF log loss {-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc)):.4f} | mean p {p.mean():.4f} vs rate {y.mean():.4f}")

### Causal decision rules
* Fixed top-1 per day ignores risk levels, costs and the budget.
* Myopic: inspect when `15 * p > hours`. It spends the budget on the first worthwhile days and
  has nothing left for later storms in the same week.
* Budget-aware: inspect only when the expected return per hour, `(15 * p - hours) / hours`, is
  above a threshold `tau`, which keeps hours for the most valuable inspections. `tau` is fixed
  in advance from validation, so each day's decision uses only that day's information.

In [ ]:
hours = tr.inspection_hours.to_numpy()
rank_in_day = lambda s: tr.assign(s=s).groupby("day")["s"].rank(ascending=False, method="first").to_numpy()
results = {
    "inspect nothing": fold_score(np.zeros(len(tr)), np.zeros(len(tr))),
    "top-1 by LR raw every day": fold_score(rank_in_day(oof["LR raw"]) <= 1, oof["LR raw"]),
}
for name, p in oof.items():
    results[f"myopic 15p > hours, {name}"] = fold_score(CATCH_VALUE * p > hours, p)

taus = np.round(np.arange(0.0, 2.01, 0.1), 2)
tau_table = {}
for name, p in oof.items():
    density = (CATCH_VALUE * p - hours) / hours
    tau_table[name] = {t: fold_score(density > t, density) for t in taus}
    t_best = max(tau_table[name], key=tau_table[name].get)
    results[f"budget-aware (tau={t_best}), {name}"] = tau_table[name][t_best]
summary = pd.Series(results, name="site-grouped weekly net value (higher is better)").round(4)
print(summary.to_string())

## Final plan
Use the probability model and `tau` with the best validation score, refit on all training
sites, and list for each test week every site-day whose return per hour exceeds `tau`, in date
order and by return within a day. The grader executes the list against the 12-hour budget.

In [ ]:
best_name = max(tau_table, key=lambda n: max(tau_table[n].values()))
tau = max(tau_table[best_name], key=tau_table[best_name].get)
print(f"chosen model: {best_name}, tau = {tau}")

def fit_predict(name):
    if name == "LR raw":
        return make_lr(BASE).fit(tr[BASE], tr[TARGET]).predict_proba(te[BASE])[:, 1]
    if name == "LR engineered":
        return make_lr(ENGINEERED).fit(tr[ENGINEERED], tr[TARGET]).predict_proba(te[ENGINEERED])[:, 1]
    n = int(np.mean(best_iters) * 1.2)
    p = np.zeros(len(te))
    for s in range(5):
        m = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": n, "random_state": SEED + s})
        p += m.fit(tr[ENGINEERED], tr[TARGET]).predict_proba(te[ENGINEERED])[:, 1] / 5
    return p if name == "LightGBM engineered" else 0.5 * p + 0.5 * fit_predict("LR raw")

te["p"] = fit_predict(best_name)
te["density"] = (CATCH_VALUE * te.p - te.inspection_hours) / te.inspection_hours
chosen = te[te.density > tau].sort_values(["day", "density"], ascending=[True, False])
tokens = (chosen.location_id.astype(str) + "@" + chosen.day).groupby(chosen.query_id).agg(" ".join)
submission = test_queries[["query_id"]].assign(site_days=lambda d: d.query_id.map(tokens).fillna(""))

assert len(submission) == len(sample) and set(submission.query_id) == set(sample.query_id)
print("listed site-days per week:", round(len(chosen) / len(submission), 2),
      "| weeks with nothing listed:", int((submission.site_days == "").sum()))
submission.to_csv(OUT / "submission.csv", index=False)
submission.head()